# LLPS Predictor — regime-aware

Paste a protein sequence, set the buffer conditions, get a prediction.

What makes this different from a normal predictor: before answering, it works out
**which generalization regime your protein falls in**, and quotes the accuracy that
actually applies to your case instead of a single headline number.

| If your sequence… | Regime | Honest accuracy |
|---|---|---|
| is in the training database | **A** — characterised protein | ROC-AUC 0.834 |
| is new, but a homolog is in the database (≥50% identity) | **B** — new protein | 0.786 |
| has no homolog in the database | **C\*** — new protein family | 0.659 |

The prediction itself is valid in every case. What changes is how much you should
trust it — and a model that reports 0.834 for a sequence unlike anything it trained on
is overstating its reliability by a wide margin.

**Setup:** Runtime → Change runtime type → **T4 GPU**.
Upload `llpsense_final_noseq.csv`, `llpsense_t5.npz` and `llpsense_sequences.csv`.
Steps 1–5 take about 6 minutes; after that each prediction is a few seconds.

## 1. Install and upload

In [ ]:
!pip -q install "transformers==4.44.2" sentencepiece parasail
import numpy as np, pandas as pd, torch, re, warnings, time
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from google.colab import files
print("GPU:", torch.cuda.is_available())
up = files.upload()        # the three files listed above

In [ ]:
df   = pd.read_csv("llpsense_final_noseq.csv")
z    = np.load("llpsense_t5.npz", allow_pickle=True)
emb  = pd.DataFrame(z["X"], index=z["SeqID"])
seqs = pd.read_csv("llpsense_sequences.csv")           # SeqID, Sequence
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
MAXV = dict(Conc=1000., pH=14., temp=60., MgCl2=50., NaCl=2000., KCl=1000.,
            **{k:50. for k in ['PEG300-1k','PEG3k-6k','PEG8k-20k','Ficoll','Dextran -40','Dextran 70-']},
            Glycerol=10.)
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y, w = df.Label.to_numpy(), df.w_record.to_numpy()
N_EMB = emb.shape[1]; IDX = {c: N_EMB+i for i,c in enumerate(COND)}
print(f"{len(df):,} condition rows | {df.SeqID.nunique()} sequences | features {X.shape[1]}")

## 2. Load ProtT5

In [ ]:
from transformers import T5Tokenizer, T5EncoderModel
MODEL = "Rostlab/prot_t5_xl_half_uniref50-enc"
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
tok = T5Tokenizer.from_pretrained(MODEL, do_lower_case=False, use_safetensors=True)
t5  = T5EncoderModel.from_pretrained(MODEL, use_safetensors=True).to(dev).eval()
t5.requires_grad_(False)

def embed(seq):
    s = " ".join(list(re.sub(r"[UZOB]", "X", seq.strip().upper())))
    t = tok([s], add_special_tokens=True, padding="longest")
    ii = torch.tensor(t["input_ids"]).to(dev); am = torch.tensor(t["attention_mask"]).to(dev)
    with torch.no_grad():
        out = t5(input_ids=ii, attention_mask=am).last_hidden_state
    return ((out*am[...,None]).sum(1)/am.sum(1,keepdim=True)).float().cpu().numpy()[0]
print("ProtT5 ready")

## 3. Train the models

Two models: the physics-constrained one used for predictions, and the unconstrained
one kept for comparison so you can see where it disagrees.

In [ ]:
from xgboost import XGBClassifier
UP = ['Conc','PEG300-1k','PEG3k-6k','PEG8k-20k','Ficoll','Dextran -40','Dextran 70-']
MONO = tuple([0]*N_EMB + [1 if c in UP else 0 for c in COND])
HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899, random_state=42)
RT = dict(tree_method="hist", eval_metric="logloss",
          **({"device":"cuda"} if torch.cuda.is_available() else {"n_jobs":-1}))
t0=time.time()
MODEL_C = XGBClassifier(**HP, monotone_constraints=MONO, **RT).fit(X, y, sample_weight=w)
MODEL_U = XGBClassifier(**HP, **RT).fit(X, y, sample_weight=w)
print(f"trained in {time.time()-t0:.0f}s")

## 4. Regime detection

Your sequence is aligned against all 854 training sequences (Smith-Waterman, BLOSUM62).
The best hit decides the regime.

A note on why this is the honest construction: if your sequence is **not** in the
training set, the model genuinely never saw it, so its prediction already *is* the
regime B or C\* prediction — nothing needs retraining. Only the reliability figure
changes. If your sequence **is** in the training set, the model has seen it, and
regime A applies.

In [ ]:
import parasail
SEQS = seqs.Sequence.tolist(); SIDS = seqs.SeqID.tolist()
NAME = df.drop_duplicates("SeqID").set_index("SeqID")["ProteinName"].to_dict()
BL = parasail.blosum62

def best_hit(q):
    best = (0.0, 0.0, None)
    for sid, s in zip(SIDS, SEQS):
        if max(len(s), len(q)) > 3*min(len(s), len(q)):   # cheap length prefilter
            continue
        tr = parasail.sw_trace_striped_16(q, s, 11, 1, BL).traceback
        al = len(tr.query)
        if not al: continue
        ident = sum(1 for a,b in zip(tr.query, tr.ref) if a==b)/al
        cov   = sum(1 for a in tr.query if a!='-')/len(q)
        if ident*cov > best[0]*best[1]:
            best = (ident, cov, sid)
    return best

REGIME_AUC = {"A":0.834, "B":0.786, "C*":0.659}
def classify(q):
    ident, cov, sid = best_hit(q)
    if sid is None:            return "C*", ident, cov, None
    if ident >= 0.99 and cov >= 0.99:  return "A",  ident, cov, sid
    if ident >= 0.50 and cov >= 0.90:  return "B",  ident, cov, sid
    return "C*", ident, cov, sid
print("regime detection ready")

## 5. Predict

Edit the sequence and the conditions, then run. Leave a condition at 0 if it is absent
from your buffer.

In [ ]:
# ---------------- YOUR INPUT ----------------
SEQUENCE = (
  "MASNDYTQQATQSYGAYPTQPGQGYSQQSSQPYGQQSYSGYSQSTDTSGYGQSSYSSYGQSQNTGYGTQSTPQGYGSTGGYGSSQSSQSSYGQQSSYPGYGQQPAPSSTSGSYGSSSQSSSYGQPQSGSYSQQPSYGGQQQSYGQQQSYNPPQGYGQQNQYNS"
)
PROTEIN_LABEL = "my sequence"

CONDITIONS = dict(
    Conc        = 20.0,    # protein concentration, uM
    pH          = 7.4,
    temp        = 25.0,    # degrees C
    NaCl        = 150.0,   # mM
    KCl         = 0.0,     # mM
    MgCl2       = 0.0,     # mM
    **{'PEG300-1k': 0.0, 'PEG3k-6k': 0.0, 'PEG8k-20k': 0.0,
       'Ficoll': 0.0, 'Dextran -40': 0.0, 'Dextran 70-': 0.0},
    Glycerol    = 0.0,     # %
)
# --------------------------------------------

def featurise(e, cond):
    v = np.zeros(len(COND), dtype=np.float32)
    for i, c in enumerate(COND):
        v[i] = float(cond.get(c, 0.0)) / MAXV[c]
    return np.hstack([e, v])[None, :]

seq = re.sub(r"[^A-Za-z]", "", SEQUENCE).upper()
t0 = time.time()
e  = embed(seq)
reg, ident, cov, hit = classify(seq)
row = featurise(e, CONDITIONS)
p_c = float(MODEL_C.predict_proba(row)[0,1])
p_u = float(MODEL_U.predict_proba(row)[0,1])

print(f"{PROTEIN_LABEL}  ({len(seq)} residues)   [{time.time()-t0:.1f}s]")
print("="*68)
print(f"  P(LLPS) = {p_c:.3f}   ->  {'phase separation predicted' if p_c>0.5 else 'no phase separation predicted'}")
print()
print(f"  Regime {reg}  —  {'in training database' if reg=='A' else ('homolog in database' if reg=='B' else 'no close homolog')}")
if hit: print(f"    closest training sequence: {NAME.get(hit,hit)}  ({100*ident:.0f}% identity, {100*cov:.0f}% coverage)")
print(f"    accuracy that applies here: ROC-AUC {REGIME_AUC[reg]:.3f}")
if reg != "A":
    print(f"    note: a single-figure predictor would quote 0.834 here. That would overstate")
    print(f"          reliability by {0.834-REGIME_AUC[reg]:.3f} ROC-AUC for this sequence.")
print()
print(f"  unconstrained model would say {p_u:.3f}  (difference {p_u-p_c:+.3f})")

## 6. Phase diagram for your sequence

In [ ]:
CONC = np.logspace(np.log10(0.5), np.log10(563.0), 40)
SALT = np.linspace(0, 1000, 26)
G = np.zeros((len(SALT), len(CONC)), dtype=np.float32)
rows = []
for s in SALT:
    for c in CONC:
        cd = dict(CONDITIONS); cd['NaCl'] = s; cd['Conc'] = c
        rows.append(featurise(e, cd)[0])
G = MODEL_C.predict_proba(np.array(rows))[:,1].reshape(len(SALT), len(CONC))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
im = ax[0].pcolormesh(CONC, SALT, G, cmap="RdBu_r", vmin=0, vmax=1, shading="auto")
ax[0].contour(CONC, SALT, G, levels=[0.5], colors="k", linewidths=2)
ax[0].set(xscale="log", xlabel="protein concentration (µM)", ylabel="NaCl (mM)",
          title=f"{PROTEIN_LABEL} — predicted phase diagram (regime {reg})")
ax[0].plot(CONDITIONS['Conc'], CONDITIONS['NaCl'], "o", ms=9, mfc="none", mec="k", mew=2)
fig.colorbar(im, ax=ax[0], label="P(LLPS)")

si = int(np.argmin(abs(SALT - CONDITIONS['NaCl'])))
ax[1].plot(CONC, G[si], lw=2.4, color="#1f5fa8", label="physics-constrained")
rows_u = [featurise(e, {**CONDITIONS, 'Conc': c})[0] for c in CONC]
ax[1].plot(CONC, MODEL_U.predict_proba(np.array(rows_u))[:,1], lw=1.8, ls="--",
           color="#9a3412", label="unconstrained")
ax[1].axhline(.5, color="k", ls=":", lw=1)
ax[1].set(xscale="log", xlabel="protein concentration (µM)", ylabel="P(LLPS)",
          title=f"response at {CONDITIONS['NaCl']:.0f} mM NaCl", ylim=(0,1))
ax[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("prediction.png", dpi=150); plt.show()

cs = CONC[np.argmax(G[si] > 0.5)] if (G[si] > 0.5).any() else None
print(f"predicted saturation concentration at these conditions: "
      f"{f'{cs:.2f} µM' if cs is not None else 'no crossing below 563 µM'}")

## 7. Several sequences at once

Paste a FASTA block to rank a set of candidates. Each gets its own regime.

In [ ]:
FASTA = """
>candidate_1
MASNDYTQQATQSYGAYPTQPGQGYSQQSSQPYGQQSYSGYSQSTDTSGYGQSSYSSYGQSQNTGYGTQSTPQGYGSTGGYGSSQSSQSSYGQ
>candidate_2
MESNQSNNGGSGNAALNRGGRFLQYLQSHPLGSLLSHLLSQAGGQHTPHHHHHHHHHGGSAAAAVEPLSKGYPQAAFIQAPPSFP
"""

recs, name, buf = [], None, []
for ln in FASTA.strip().splitlines():
    ln = ln.strip()
    if ln.startswith(">"):
        if name: recs.append((name, "".join(buf)))
        name, buf = ln[1:].strip(), []
    elif ln:
        buf.append(re.sub(r"[^A-Za-z]", "", ln).upper())
if name: recs.append((name, "".join(buf)))

out = []
for nm, sq in recs:
    ee = embed(sq); rg, idn, cv, h = classify(sq)
    pp = float(MODEL_C.predict_proba(featurise(ee, CONDITIONS))[0,1])
    out.append({"sequence": nm, "length": len(sq), "P(LLPS)": round(pp,3),
                "regime": rg, "applies_AUC": REGIME_AUC[rg],
                "closest_match": NAME.get(h, h) if h else "—",
                "identity_%": round(100*idn)})
res = pd.DataFrame(out).sort_values("P(LLPS)", ascending=False)
print(res.to_string(index=False))
res.to_csv("predictions.csv", index=False)

### Reading the output

A prediction of 0.8 under regime C\* and a prediction of 0.8 under regime A are not the
same claim. The first comes from a model working on a protein family it has never seen,
where measured accuracy is 0.659 — barely above the 0.548 majority baseline. The second
comes from a model that has seen this protein under other conditions, where accuracy is
0.834 and the probabilities are well calibrated.

Reporting which of those situations a user is in is the contribution. It costs nothing
to compute and no published condition-aware LLPS predictor currently does it.